# BODAQS Batch Preprocessor - Self-scoped

This notebook processes selected logger session inputs into one configured BODAQS library. It creates one run per requested batch, writes directly to the library root, and does not require an Import Manager source.

## 1. Configure Library And Controls

Set `LIBRARIES_ROOT` and `LIBRARY_ID`, then run this cell. Use the controls to choose runtime paths, select input files, and optionally attach draft notes.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import ipywidgets as W
import pandas as pd


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / "OneDrive" / "BODAQS-data"
LIBRARY_ID = "archie"

DEFAULT_INPUT_DIR = Path.home() / "OneDrive" / "BODAQS-data" / "sources"
DEFAULT_PREPROCESS_PROFILE_PATH = ANALYSIS_DIR / "config" / "preprocess_profiles" / "suspension_default_v1.json"
DEFAULT_BIKE_PROFILE_PATH = ANALYSIS_DIR / "config" / "bike_profiles" / "example_enduro_bike_v1.json"
DEFAULT_LOG_METADATA_PATHS = [ANALYSIS_DIR / "config" / "log_metadata_examples"]
DEFAULT_FIT_DIR = Path.home() / "OneDrive" / "BODAQS-data" / "sources" / "ben-stevo-local" / "fit"
DEFAULT_FIT_BINDINGS_PATH = ANALYSIS_DIR / "config" / "fit_bindings_v1.json"
DEFAULT_SESSION_NOTE_TEMPLATE_PATH = ANALYSIS_DIR / "templates" / "session_note_templates" / "suspension_setup" / "1.0.json"

from bodaqs_analysis.library_api import LibraryAdapter
from bodaqs_analysis.ui import make_preprocess_runtime_settings_editor
from bodaqs_analysis.ui.preprocess_file_selector import PreprocessLogSelector

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]["root"])

runtime_settings_editor = make_preprocess_runtime_settings_editor(
    log_dir=DEFAULT_INPUT_DIR,
    artifacts_dir=LIBRARY_ROOT,
    preprocess_profile_path=DEFAULT_PREPROCESS_PROFILE_PATH,
    bike_profile_path=DEFAULT_BIKE_PROFILE_PATH,
    generic_log_metadata_paths=DEFAULT_LOG_METADATA_PATHS,
    fit_dir=DEFAULT_FIT_DIR,
    fit_bindings_path=DEFAULT_FIT_BINDINGS_PATH,
    prompt_for_descriptions=True,
    run_tz_label="AWST",
    logger_timezone="Australia/Perth",
    show_log_dir=True,
    show_artifacts_dir=False,
)

selector = PreprocessLogSelector(
    artifacts_dir=LIBRARY_ROOT,
    state_file=ANALYSIS_DIR / ".bodaqs_preprocess_self_scoped_last_dir.json",
    sha_cache_file=ANALYSIS_DIR / ".bodaqs_preprocess_self_scoped_sha_cache.json",
    include_zip_archives=True,
    include_bdq_files=True,
)
settings = runtime_settings_editor.get_settings()
if settings.get("log_dir") is not None:
    selector.w_dir.value = str(settings["log_dir"])
runtime_settings_editor.bind_log_selector(selector)

attach_draft_note = W.Checkbox(value=False, description="Attach draft notes from template")
session_note_template_path = W.Text(
    value=str(DEFAULT_SESSION_NOTE_TEMPLATE_PATH),
    description="Note template",
    layout=W.Layout(width="100%"),
)
run_description = W.Text(
    value="Manual preprocessing batch",
    description="Run desc",
    layout=W.Layout(width="100%"),
)
manual_input_paths = W.Textarea(
    value="",
    description="Extra inputs",
    placeholder="Optional: one explicit CSV, ZIP, or BDQ path per line. These are added to the selector files.",
    layout=W.Layout(width="100%", height="90px"),
)

note_panel = W.VBox(
    [
        W.HTML("<h3 style='margin: 16px 0 6px 0'>Batch Options</h3>"),
        run_description,
        manual_input_paths,
        attach_draft_note,
        session_note_template_path,
    ],
    layout=W.Layout(width="100%"),
)

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Library root: {LIBRARY_ROOT}")
display(runtime_settings_editor.ui)
display(selector.ui)
display(note_panel)


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Library root: C:\Users\benco\OneDrive\BODAQS-data\libraries\archie


## 2. Preview Selected Inputs

Run this cell before processing to confirm exactly which inputs will be processed. Unlike Import Manager source imports, the batch processor always processes the requested inputs.

In [2]:
def _extra_input_paths_from_text(text: str):
    return [Path(line.strip()).expanduser() for line in (text or "").splitlines() if line.strip()]


selected_input_paths = list(selector.get_selected_files())
selected_input_paths.extend(_extra_input_paths_from_text(manual_input_paths.value))
selected_input_paths = sorted({Path(path).resolve() for path in selected_input_paths})

if not selected_input_paths:
    raise ValueError("No inputs selected. Select files in the log selector or add explicit paths under Extra inputs.")

display(pd.DataFrame({"input_path": [str(path) for path in selected_input_paths]}))


,input_path
0,C:\Users\benco\OneDrive\BODAQS-private\Test\At...


## 3. Run Batch

This cell writes artifacts into the configured library. If description prompting is enabled, it asks for optional run/session descriptions after processing.

In [3]:
from bodaqs_analysis.artifacts import ArtifactStore, set_run_description, set_session_description
from bodaqs_analysis.library_preprocessing import (
    PreprocessBatchRequest,
    preprocess_requested_sessions_to_library,
)

runtime_settings = runtime_settings_editor.get_settings()
errors, warnings = runtime_settings_editor.validate(print_to_output=False)
if errors:
    raise ValueError("Runtime settings have blocking errors: " + "; ".join(errors))
for warning in warnings:
    print(f"Warning: {warning}")

selected_input_paths = list(selector.get_selected_files())
selected_input_paths.extend(_extra_input_paths_from_text(manual_input_paths.value))
selected_input_paths = sorted({Path(path).resolve() for path in selected_input_paths})
if not selected_input_paths:
    raise ValueError("No inputs selected. Select files in the log selector or add explicit paths under Extra inputs.")

profile_path = Path(runtime_settings["preprocess_profile_path"])
bike_profile_path = runtime_settings.get("bike_profile_path")
if bike_profile_path is None:
    raise ValueError("Bike profile path is blank.")

note_template = Path(session_note_template_path.value).expanduser() if attach_draft_note.value else None


def progress(event, payload):
    if event in {"input_started", "input_succeeded", "input_failed"}:
        print(f"{event}: {payload.get('input_path')} {payload.get('session_id', '')} {payload.get('error', '')}")


batch_result = preprocess_requested_sessions_to_library(
    PreprocessBatchRequest(
        artifacts_dir=LIBRARY_ROOT,
        input_paths=tuple(selected_input_paths),
        preprocess_profile_path=profile_path,
        bike_profile_path=Path(bike_profile_path),
        run_tz_label=runtime_settings["run_tz_label"],
        run_description=run_description.value.strip() or None,
        generic_log_metadata_paths=tuple(runtime_settings.get("generic_log_metadata_paths") or ()),
        log_metadata_path=None,
        fit_dir=runtime_settings.get("fit_dir"),
        fit_bindings_path=runtime_settings.get("fit_bindings_path"),
        logger_timezone=runtime_settings.get("logger_timezone"),
        include_events=True,
        include_metrics=True,
        attach_draft_note=bool(attach_draft_note.value),
        session_note_template_path=note_template,
        continue_on_error=True,
    ),
    progress_callback=progress,
)

if runtime_settings.get("prompt_for_descriptions"):
    store = ArtifactStore(LIBRARY_ROOT)
    run_desc_default = run_description.value.strip()
    entered = input(f"Run description for {batch_result['run_id']} [{run_desc_default}]: ").strip()
    if entered or run_desc_default:
        set_run_description(store, run_id=batch_result["run_id"], description=entered or run_desc_default)
    for item in batch_result["results"]:
        if item.get("status") != "succeeded":
            continue
        sid = str(item["session_id"])
        s_desc = input(f"Session description for {sid} (blank to skip): ").strip()
        if s_desc:
            set_session_description(store, run_id=batch_result["run_id"], session_id=sid, description=s_desc)

selector.refresh()

print(f"Run written: {batch_result['run_id']}")
print(f"Run manifest: {batch_result['run_manifest_path']}")
display(pd.DataFrame(batch_result["results"]))


input_started: C:\Users\benco\OneDrive\BODAQS-private\Test\Attitude\260820_095200.bdq  


Logger firmware reported dropped samples: samples_dropped=469 csv=C:\Users\benco\OneDrive\BODAQS-private\Test\Attitude\260820_095200.bdq
C:\Users\benco\dev\BODAQS\analysis\bodaqs_analysis\metrics.py:617: RuntimeWarning: Mean of empty slice
  out[r] = float(reducer(y[r, a:b]))
C:\Users\benco\dev\BODAQS\analysis\bodaqs_analysis\metrics.py:617: RuntimeWarning: All-NaN slice encountered
  out[r] = float(reducer(y[r, a:b]))


input_succeeded: C:\Users\benco\OneDrive\BODAQS-private\Test\Attitude\260820_095200.bdq 260820_095200 


Run description for run_2026-08-20T11-28-33_AWST [Manual preprocessing batch]:  
Session description for 260820_095200 (blank to skip):  


Run written: run_2026-08-20T11-28-33_AWST
Run manifest: C:\Users\benco\OneDrive\BODAQS-data\libraries\archie\runs\run_2026-08-20T11-28-33_AWST\manifest.json


,status,input_path,run_id,session_id,session_key,session_manifest_path,events_written,metrics_written
0,succeeded,C:\Users\benco\OneDrive\BODAQS-private\Test\At...,run_2026-08-20T11-28-33_AWST,260820_095200,run_2026-08-20T11-28-33_AWST::260820_095200,C:\Users\benco\OneDrive\BODAQS-data\libraries\...,"[Jump, compressions_all, rebounds_all]","[Jump, compressions_all, rebounds_all]"
